# Phase 3.1 — Targeted QA repair

This notebook repairs only the 21 pages flagged by the signed Phase 3 run. It
preserves Phase 3, creates a curated 408-page overlay, and excludes any page
that still fails the quality gate from retrieval.

Run **Runtime → Run all**. Do not unzip the package manually. No transcription
or LLM API key is required.


## 1. Mount the known clean project


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

EXPECTED_PACKAGE_SHA256 = "def5fb11aa854fc1cc525f46883cdf8a8ffba864187b380cb765a28ec141030c"
PACKAGE_FILENAME = "PHASE_3_1_TARGETED_QA_REPAIR_PACKAGE.zip"

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = Path("/content/drive/MyDrive/Devoteam internship/Devoteam_AI_CLEAN_PIPELINE")
assert (PROJECT_ROOT / "config" / "project.yaml").exists(), f"Clean project not found: {PROJECT_ROOT}"
assert (PROJECT_ROOT / "data" / "extracted" / "20260714T154731Z_129ff982c8" / "phase3_extract_v1" / "_SUCCESS.json").exists(), "Signed Phase 3 result is missing"
PACKAGE_PATH = PROJECT_ROOT / PACKAGE_FILENAME
assert PACKAGE_PATH.exists(), f"Missing package: {PACKAGE_PATH}"
print(f"Project root: {PROJECT_ROOT}")


## 2. Verify and install the signed additive package


In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

actual_package_sha = file_sha256(PACKAGE_PATH)
assert actual_package_sha == EXPECTED_PACKAGE_SHA256, "Phase 3.1 package hash mismatch"

with zipfile.ZipFile(PACKAGE_PATH) as archive:
    for member in archive.infolist():
        target = (PROJECT_ROOT / member.filename).resolve()
        assert str(target).startswith(str(PROJECT_ROOT.resolve()) + os.sep), f"Unsafe archive path: {member.filename}"
    package_manifest = json.loads(archive.read("PHASE_3_1_PACKAGE_MANIFEST.json"))
    for entry in package_manifest["files"]:
        assert hashlib.sha256(archive.read(entry["path"])).hexdigest() == entry["sha256"]
    for member in archive.infolist():
        if member.is_dir() or member.filename == "PHASE_3_1_PACKAGE_MANIFEST.json":
            continue
        destination = PROJECT_ROOT / member.filename
        packaged_hash = hashlib.sha256(archive.read(member.filename)).hexdigest()
        if destination.exists():
            assert file_sha256(destination) == packaged_hash, f"Refusing to overwrite changed file: {member.filename}"
        else:
            archive.extract(member, PROJECT_ROOT)

print(f"Verified package SHA-256: {actual_package_sha}")
print("Phase 3.1 additive extension installed safely.")


## 3. Check OCR dependencies and run the full project tests


In [ ]:
language_result = subprocess.run(
    ["tesseract", "--list-langs"], text=True, capture_output=True
) if shutil.which("tesseract") else None
installed_languages = set(language_result.stdout.splitlines()[1:]) if language_result and language_result.returncode == 0 else set()
missing_languages = {"eng", "fra", "ara"} - installed_languages
if missing_languages:
    print(f"Installing OCR language packs {sorted(missing_languages)} — usually 1–3 minutes...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "tesseract-ocr", "tesseract-ocr-fra", "tesseract-ocr-eng", "tesseract-ocr-ara"],
        check=True,
    )
else:
    print("Tesseract French/English/Arabic language packs are available.")

print("Installing/checking Python packages — usually 1–3 minutes...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements" / "phase3_1.txt")],
    check=True,
)
environment = os.environ.copy()
environment["PYTHONPATH"] = str(PROJECT_ROOT / "src")
tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", str(PROJECT_ROOT / "tests")],
    cwd=PROJECT_ROOT,
    env=environment,
    text=True,
    capture_output=True,
)
print(tests.stdout)
if tests.stderr:
    print(tests.stderr)
assert tests.returncode == 0, "Tests failed; targeted repair did not start."
print("All foundation, snapshot, extraction, and repair tests passed.")


## 4. Run or resume the 21-page targeted repair


In [ ]:
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from devoteam_reference_ai.phase3_1_repair import run_phase3_1

summary = run_phase3_1(
    project_root=PROJECT_ROOT,
    config_path=PROJECT_ROOT / "config" / "phase3_1_repair.yaml",
    progress=print,
)
print(json.dumps(summary, indent=2, ensure_ascii=False, sort_keys=True))


## 5. Verify hashes and publish the Phase 3.1 gate


In [ ]:
from devoteam_reference_ai.phase3_1_repair import verify_phase3_1

assert summary["status"] == "PASS", "Targeted repair had processing errors; rerun to retry only incomplete pages."
verified = verify_phase3_1(Path(summary["run_root"]))
assert verified["pages_targeted"] == 21
assert verified["curated_pages_total"] == 408
assert verified["repair_processing_failures"] == 0
assert verified["source_snapshot_mutation_calls"] == 0
assert verified["phase3_output_mutation_calls"] == 0
assert verified["external_llm_calls"] == 0

print("PHASE 3.1: TECHNICAL PASS")
print(f"Pages repaired to PASS: {verified['pages_repaired_to_pass']}/21")
print(f"Pages still REVIEW: {verified['pages_still_review']}")
print(f"Pages still FAILED: {verified['pages_still_failed']}")
print(f"Retrieval-eligible pages: {verified['retrieval_eligible_pages']}/408")
print(f"QA gate: {verified['qa_gate']}")
print("Send this final block for senior review before Phase 4.")
